# Data Processing

In [1]:
DATA_DIR = "../data"
DATASET_DIR = f"{DATA_DIR}/processed/csecicids2018"

In [2]:
import os

for dirname, _, filenames in os.walk(DATASET_DIR):
    for filename in filenames:
        print(os.path.join(dirname, filename))

../data/processed/csecicids2018\Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet
../data/processed/csecicids2018\Bruteforce-Wednesday-14-02-2018_TrafficForML_CICFlowMeter.parquet
../data/processed/csecicids2018\DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMeter.parquet
../data/processed/csecicids2018\DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlowMeter.parquet
../data/processed/csecicids2018\DoS1-Thursday-15-02-2018_TrafficForML_CICFlowMeter.parquet
../data/processed/csecicids2018\DoS2-Friday-16-02-2018_TrafficForML_CICFlowMeter.parquet
../data/processed/csecicids2018\Infil1-Wednesday-28-02-2018_TrafficForML_CICFlowMeter.parquet
../data/processed/csecicids2018\Infil2-Thursday-01-03-2018_TrafficForML_CICFlowMeter.parquet
../data/processed/csecicids2018\Web1-Thursday-22-02-2018_TrafficForML_CICFlowMeter.parquet
../data/processed/csecicids2018\Web2-Friday-23-02-2018_TrafficForML_CICFlowMeter.parquet


In [3]:
import pandas as pd
import pprint as pp

In [4]:
parquet_collection = []

# read parquet files into pandas dataframes
for dirname, _, filenames in os.walk(DATASET_DIR):
    for filename in filenames:
        if filename.endswith(".parquet"):
            path = os.path.join(dirname, filename)
            print(f"Reading {path}")
            # df = pd.read_parquet(path).sample(frac=0.2, replace=False, random_state=42)
            df = pd.read_parquet(path)
            parquet_collection.append(df)

Reading ../data/processed/csecicids2018\Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet
Reading ../data/processed/csecicids2018\Bruteforce-Wednesday-14-02-2018_TrafficForML_CICFlowMeter.parquet
Reading ../data/processed/csecicids2018\DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMeter.parquet
Reading ../data/processed/csecicids2018\DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlowMeter.parquet
Reading ../data/processed/csecicids2018\DoS1-Thursday-15-02-2018_TrafficForML_CICFlowMeter.parquet
Reading ../data/processed/csecicids2018\DoS2-Friday-16-02-2018_TrafficForML_CICFlowMeter.parquet
Reading ../data/processed/csecicids2018\Infil1-Wednesday-28-02-2018_TrafficForML_CICFlowMeter.parquet
Reading ../data/processed/csecicids2018\Infil2-Thursday-01-03-2018_TrafficForML_CICFlowMeter.parquet
Reading ../data/processed/csecicids2018\Web1-Thursday-22-02-2018_TrafficForML_CICFlowMeter.parquet
Reading ../data/processed/csecicids2018\Web2-Friday-23-02-2018_TrafficForML_CICFlowMeter.parquet


## Convert the Dataset from Multi-Label to Binary Classification

In [4]:
df = pd.read_parquet(f"{DATA_DIR}/processed/cse-binary.parquet")
df.head(5)

,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,6,141385,9,7,553,3773.0,202,0,61.444443,87.534439,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1,6,281,2,1,38,0.0,38,0,19.000000,26.870058,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
2,6,279824,11,15,1086,10527.0,385,0,98.727272,129.392502,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
3,6,132,2,0,0,0.0,0,0,0.000000,0.000000,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
4,6,274016,9,13,1285,6141.0,517,0,142.777771,183.887726,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign


In [5]:
df.shape

(6659532, 78)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6659532 entries, 0 to 6659531
Data columns (total 78 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   Protocol                  int8   
 1   Flow Duration             int64  
 2   Total Fwd Packets         int32  
 3   Total Backward Packets    int32  
 4   Fwd Packets Length Total  int32  
 5   Bwd Packets Length Total  float64
 6   Fwd Packet Length Max     int32  
 7   Fwd Packet Length Min     int16  
 8   Fwd Packet Length Mean    float32
 9   Fwd Packet Length Std     float32
 10  Bwd Packet Length Max     int32  
 11  Bwd Packet Length Min     int16  
 12  Bwd Packet Length Mean    float32
 13  Bwd Packet Length Std     float32
 14  Flow Bytes/s              float64
 15  Flow Packets/s            float64
 16  Flow IAT Mean             float32
 17  Flow IAT Std              float32
 18  Flow IAT Max              float64
 19  Flow IAT Min              float64
 20  Fwd IAT Total           

## Select KBest Features

In [7]:
X = df.drop(columns=["Label"])
y = df["Label"]

In [8]:
X.shape, y.shape

((6659532, 77), (6659532,))

In [9]:
# remove columns with only one unique value
X = X.loc[:, X.nunique() != 1]

X.shape

(6659532, 69)

In [10]:
from sklearn.feature_selection import SelectKBest, f_classif


def select_k_best_features(X, y, k=10):
    selector = SelectKBest(score_func=f_classif, k=k)
    X_new = selector.fit_transform(X, y)
    mask = selector.get_support()
    selected_features = X.columns[mask]
    return selected_features


selected_features = select_k_best_features(X, y, k=64)

selected_features.shape


(64,)

In [11]:
# list of selected features
selected_features



Index(['Protocol', 'Flow Duration', 'Total Fwd Packets',
       'Total Backward Packets', 'Fwd Packets Length Total',
       'Bwd Packets Length Total', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Std', 'Flow IAT Max', 'Fwd IAT Total', 'Fwd IAT Std',
       'Fwd IAT Max', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd URG Flags',
       'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s',
       'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max',
       'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance',
       'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count',
       'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count',
     

In [12]:
# list unselected features
unselected_features = X.columns.difference(selected_features)
unselected_features

Index(['Flow IAT Mean', 'Flow IAT Min', 'Fwd IAT Mean', 'Fwd IAT Min',
       'Idle Std'],
      dtype='object')

In [13]:
X_selected_features = X[selected_features]


X_selected_features.head()

,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Init Bwd Win Bytes,Fwd Act Data Packets,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Max,Idle Min
0,6,141385,9,7,553,3773.0,202,0,61.444443,87.534439,...,119,4,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,6,281,2,1,38,0.0,38,0,19.000000,26.870058,...,0,0,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,6,279824,11,15,1086,10527.0,385,0,98.727272,129.392502,...,1047,5,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,6,132,2,0,0,0.0,0,0,0.000000,0.000000,...,-1,0,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,6,274016,9,13,1285,6141.0,517,0,142.777771,183.887726,...,1047,5,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
X_selected_features.shape

(6659532, 64)

## Encoding fields

In [15]:
# get dtype and unique values of each column -> print
for column in X_selected_features.columns:

    pp.pprint(
        {
            "column": column,
            "dtype": X_selected_features[column].dtype,
            "unique_values": X_selected_features[column].nunique(),
        }
    )


{'column': 'Protocol', 'dtype': dtype('int8'), 'unique_values': 3}
{'column': 'Flow Duration', 'dtype': dtype('int64'), 'unique_values': 3161639}
{'column': 'Total Fwd Packets', 'dtype': dtype('int32'), 'unique_values': 4565}
{'column': 'Total Backward Packets',
 'dtype': dtype('int32'),
 'unique_values': 2968}
{'column': 'Fwd Packets Length Total',
 'dtype': dtype('int32'),
 'unique_values': 16899}
{'column': 'Bwd Packets Length Total',
 'dtype': dtype('float64'),
 'unique_values': 64903}
{'column': 'Fwd Packet Length Max',
 'dtype': dtype('int32'),
 'unique_values': 1498}
{'column': 'Fwd Packet Length Min',
 'dtype': dtype('int16'),
 'unique_values': 410}
{'column': 'Fwd Packet Length Mean',
 'dtype': dtype('float32'),
 'unique_values': 68675}
{'column': 'Fwd Packet Length Std',
 'dtype': dtype('float32'),
 'unique_values': 172590}
{'column': 'Bwd Packet Length Max',
 'dtype': dtype('int32'),
 'unique_values': 1462}
{'column': 'Bwd Packet Length Min',
 'dtype': dtype('int16'),
 'uniq

In [16]:
# print all column names
X_selected_features.columns

Index(['Protocol', 'Flow Duration', 'Total Fwd Packets',
       'Total Backward Packets', 'Fwd Packets Length Total',
       'Bwd Packets Length Total', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Std', 'Flow IAT Max', 'Fwd IAT Total', 'Fwd IAT Std',
       'Fwd IAT Max', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd URG Flags',
       'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s',
       'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max',
       'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance',
       'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count',
       'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count',
     